# Gợi ý lọc dựa trên nội dung - Phương pháp Heuristic

## Introduction

Phương pháp heuristic sẽ tìm các POI mà người dùng mục tiêu đã ghé thăm, tìm cùng danh mục, hoặc trong danh mục đó, tìm các POI mà người dùng chưa ghé thăm, và tìm các POI hàng đầu có số lượng đánh giá nhiều nhất.

Đối với mỗi POI, đếm số lần xuất hiện trong cả danh sách khu vực và danh mục.
Kết hợp các số đếm này thành một số đo duy nhất gọi là 'total_weight'. Trọng số tổng hợp này đại diện cho tần suất POI xuất hiện trên cả danh sách khu vực và danh mục. Xếp hạng các POI dựa trên tổng trọng số của chúng theo thứ tự giảm dần. Nếu tổng trọng số bằng nhau, xếp hạng sẽ dựa trên số lần xuất hiện.

Số đo này nhấn mạnh các POI thường được tìm thấy trong cả danh sách khu vực và danh mục, cho thấy mức độ liên quan hoặc tương đồng cao hơn với các POI đã đánh giá của người dùng. Hướng tiếp cận này tập trung vào tần suất xuất hiện trong các bối cảnh khác nhau (danh sách khu vực và danh mục) để ưu tiên các gợi ý.

## Điều kiện tiên quyết

Phiên bản cơ sở dữ liệu `neo4j` phải đã được khởi tạo và điền dữ liệu.

Thông tin kết nối `HOST`, `DATABASE` và `PASSWORD` phải được lưu trữ trong `NEO4J_CONF_FILE` để thiết lập kết nối tới cơ sở dữ liệu neo4j.

Sử dụng `neo4j python driver` để truy vấn cơ sở dữ liệu neo4j.

Sử dụng truy vấn `Cypher` để tạo các gợi ý.

In [33]:
import os
import configparser
import textwrap
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from neo4j import GraphDatabase

In [34]:
# Sử dụng file ini cho thông tin đăng nhập, nếu không sử dụng mặc định
HOST = 'neo4j://localhost'
DATABASE = 'neo4j'
PASSWORD = 'password'

NEO4J_CONF_FILE = 'neo4j.ini'

if NEO4J_CONF_FILE is not None and os.path.exists(NEO4J_CONF_FILE):
    config = configparser.RawConfigParser()
    config.read(NEO4J_CONF_FILE)
    HOST = config['NEO4J']['HOST']
    DATABASE = config['NEO4J'].get('DATABASE', 'neo4j')
    USERNAME = config['NEO4J'].get('USERNAME', DATABASE)
    PASSWORD = config['NEO4J']['PASSWORD']
    print('Using custom database properties')
else:
    print('Could not find database properties file, using defaults')


# Kết nối bằng neo4j python driver
driver = GraphDatabase.driver(HOST, auth=(USERNAME, PASSWORD))

Using custom database properties


In [35]:
# Hàm trợ giúp (helper)
def run(driver, query, params=None):
    with driver.session(database=DATABASE) as session:
        if params is not None:
            return [r for r in session.run(query, params)]
        else:
            return [r for r in session.run(query)]

## Gợi ý bằng Heuristic

Dựa trên POI mà người dùng đã đánh giá trước đó, gợi ý các POI khác trong cùng danh mục hoặc khu vực.

In [36]:
# HÀM: Tạo gợi ý dựa trên phương pháp Lọc dựa trên nội dung - Heuristic
# INPUT: user_id, poi_id
# OUTPUT: dataframe[user_id, poi_id, rec_poi_id, rec_poi_name]


# Thực nghiệm lần 1:
# Precision Score: 0.01268724954952532
# Recall Score: 1.0
# Coverage Score: 0.5375284306292646
# F1 Score: 0.02505659976 

# => Bổ sung {k}, chỉ lấy top {k} poi gợi ý thay vì lấy toàn bộ để cải thiện precision và F1 Score

# Thực nghiệm lần 2 (k = 5):
# Precision Score: 0.22285714285714286
# Recall Score: 0.06200317965023847
# Coverage Score: 0.0401819560272934
# F1 Score: 0.09701492537313432

# Thực nghiệm lần 3 (k = 10):
# Precision Score: 0.1838095238095238
# Recall Score: 0.10227874933757286
# Coverage Score: 0.06557998483699773
# F1 Score: 0.13142662580864828

# Crawl lại bộ dữ liệu lần 2 (Bổ sung field description)
# Thực nghiệm lần 1:
# Precision Score: 0.01442657281638114
# Recall Score: 0.6850828729281768
# Coverage Score: 0.3851473241130487
# F1 Score: 0.02825808289417462

# Thực nghiệm lần 2 (k = 5):
# Precision Score: 0.09230769230769231
# Recall Score: 0.03314917127071823
# Coverage Score: 0.03096812988574865
# F1 Score: 0.048780487804878044

# Thực nghiệm lần 3 (k = 10):
# Precision Score: 0.07788461538461539
# Recall Score: 0.05593922651933702
# Coverage Score: 0.04720384846662658
# F1 Score: 0.06511254019292605

# Thực nghiệm lần 4 (k = 10, bổ sung tọa độ để gợi ý địa điểm lân cận):
# Precision Score: 0.08557692307692308
# Recall Score: 0.06146408839779006
# Coverage Score: 0.059230306674684305
# F1 Score: 0.07154340836012862

def heuristic_recommendation(user_id, poi_id, k=10):
    # Lấy các POI trong cùng khu vực với POI đã được người dùng đánh giá
    records_region = run(driver, textwrap.dedent("""\
        MATCH (user {id: $user_id})-[:REVIEWED]->(poi:Poi {id: $poi_id})-[:LOCATED_AT]->(region:Region)<-[:LOCATED_AT]-(other_poi:Poi)<-[rated:RATED]-(review:Review)
        WHERE poi <> other_poi
        WITH user, poi, other_poi, region, count(DISTINCT rated) AS num_reviews
        RETURN user.id AS user_id, poi.id AS poi_id, other_poi.id AS rec_poi_id, other_poi.name AS rec_poi_name, region.name AS region, num_reviews AS occurrences
        """),
        params = {'user_id': user_id, 'poi_id': poi_id}
    )
    print(f"Tìm thấy {len(records_region)} records POI có CÙNG KHU VỰC.")
    # Lấy các POI trong cùng danh mục với POI đã được người dùng đánh giá 
    records_category = run(driver, textwrap.dedent("""\
        MATCH (user {id: $user_id})-[:REVIEWED]->(poi:Poi {id: $poi_id})-[:BELONGS_TO]->(category:Category)<-[:BELONGS_TO]-(other_poi:Poi)<-[rated:RATED]-(review:Review)
        WHERE poi <> other_poi
        WITH user, poi, other_poi, category, count(DISTINCT rated) AS num_reviews
        RETURN user.id AS user_id, poi.id AS poi_id, other_poi.id AS rec_poi_id, other_poi.name AS rec_poi_name, category.name AS category_name, num_reviews AS occurrences
        """),
        params = {'user_id': user_id, 'poi_id': poi_id}
    )
    print(f"Tìm thấy {len(records_category)} records POI có CÙNG DANH MỤC.")
    # Lấy các POI lân cận (NEARBY - trong bán kính 1.5km)
    records_nearby = run(driver, textwrap.dedent("""\
        MATCH (user {id: $user_id})-[:REVIEWED]->(poi:Poi {id: $poi_id})-[n:NEARBY]->(other_poi:Poi)<-[rated:RATED]-(review:Review)
        WHERE poi <> other_poi
        WITH user, poi, other_poi, n.distance_km AS distance, count(DISTINCT rated) AS num_reviews
        RETURN user.id AS user_id, poi.id AS poi_id, other_poi.id AS rec_poi_id, other_poi.name AS rec_poi_name, distance, num_reviews AS occurrences
        """),
        params = {'user_id': user_id, 'poi_id': poi_id}
    )
    print(f"Tìm thấy {len(records_nearby)} records POI LÂN CẬN (< 1.5km).")
    
    
    # Convert kết quả sang DataFrame và gom nhóm tính trọng số (weight)
    if records_region:
        df_records_region = pd.DataFrame([dict(record) for record in records_region])
        # Group by 'poi_id', 'poi_name', and 'occurrences', then aggregate the count of occurrences
        df_records_region_agg = df_records_region.groupby(['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences']).size().reset_index(name='weight_region')
        print(f"[Khu vực] Rút gọn còn {len(df_records_region_agg)} records sau khi gom nhóm.")
    else:
        df_records_region_agg = pd.DataFrame(columns=['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences', 'weight_region'])
        print(f"[Khu vực] Không tìm thấy record nào.")

    if records_category:
        df_records_category = pd.DataFrame([dict(record) for record in records_category])
        # Group by 'poi_id', 'poi_name', and 'occurrences', then aggregate the count of occurrences
        df_records_category_agg = df_records_category.groupby(['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences']).size().reset_index(name='weight_category')
        print(f" [Danh mục] Rút gọn còn {len(df_records_category_agg)} records (tính trùng lặp thể loại).")
    else:
       df_records_category_agg = pd.DataFrame(columns=['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences', 'weight_category'])
       print(f"[Danh mục] Không tìm thấy record nào.")
    if records_nearby:
        df_nearby = pd.DataFrame([dict(r) for r in records_nearby])
        df_nearby_agg = df_nearby.groupby(['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences']).size().reset_index(name='weight_nearby')
    else:
        df_nearby_agg = pd.DataFrame(columns=['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences', 'weight_nearby'])
    

    df_records_region_agg.rename(columns={'user_id': 'user_id_region', 'poi_id': 'poi_id_region', 'occurrences': 'occurrences_region'}, inplace=True)
    df_records_category_agg.rename(columns={'user_id': 'user_id_category', 'poi_id': 'poi_id_category', 'occurrences': 'occurrences_category'}, inplace=True)
    df_nearby_agg.rename(columns={'user_id': 'user_id_nearby', 'poi_id': 'poi_id_nearby', 'occurrences': 'occurrences_nearby'}, inplace=True)

    # Tính tần suất xuất hiện của POI trong cả hai danh sách
    # Gộp DataFrame dựa trên 'rec_poi_id'
    recommended_interactions = pd.merge(df_records_region_agg, df_records_category_agg, on=['rec_poi_id', 'rec_poi_name'], suffixes=('_region', '_category'), how='outer')
    recommended_interactions = pd.merge(recommended_interactions, df_nearby_agg, on=['rec_poi_id', 'rec_poi_name'], suffixes=('', '_nearby'), how='outer')
    print(f"Gộp 3 danh sách (Outer Join): Tổng cộng có {len(recommended_interactions)} records.")

     # Điền giá trị fallback cho user_id, poi_id, occurrences
    recommended_interactions['user_id'] = recommended_interactions['user_id_region'].fillna(recommended_interactions['user_id_category']).fillna(recommended_interactions['user_id_nearby']).fillna(user_id)
    recommended_interactions['poi_id'] = recommended_interactions['poi_id_region'].fillna(recommended_interactions['poi_id_category']).fillna(recommended_interactions['poi_id_nearby']).fillna(poi_id)
    recommended_interactions['occurrences'] = recommended_interactions['occurrences_region'].fillna(recommended_interactions['occurrences_category']).fillna(recommended_interactions['occurrences_nearby']).fillna(0)

    # Điền các giá trị NaN bằng 0 cho các cột 'weight'
    recommended_interactions['weight_region'] = recommended_interactions['weight_region'].fillna(0)
    recommended_interactions['weight_category'] = recommended_interactions['weight_category'].fillna(0)
    recommended_interactions['weight_nearby'] = recommended_interactions['weight_nearby'].fillna(0)

    print(f"Số record VỪA cùng [khu vực] VỪA cùng [danh mục]: {len(recommended_interactions)}")
    
    # Cộng các cột 'weight' để tính tổng trọng số
    recommended_interactions['total_weight'] = recommended_interactions['weight_region'] + recommended_interactions['weight_category'] + recommended_interactions['weight_nearby']

    # Loại bỏ các cột 'weight' riêng lẻ nếu cần
    recommended_interactions.drop(['user_id_category', 'user_id_nearby', 
                                    'poi_id_category', 'poi_id_nearby', 
                                    'occurrences_category', 'occurrences_nearby', 
                                    'weight_region', 'weight_category', 'weight_nearby'], axis=1, inplace=True, errors='ignore')
    # Sắp xếp DataFrame theo 'total_weight' giảm dần, sau đó theo 'occurrences'
    recommended_interactions = recommended_interactions.sort_values(by=['total_weight', 'occurrences'], ascending=[False, False])

     # In ra Top 5 địa điểm được đề xuất hàng đầu
    print(f"*** Top {k} địa điểm được khuyến nghị:")
    for idx, row in recommended_interactions.head(k).iterrows():
        print(f"      - Hạng {idx+1}: POI ID = {row['rec_poi_id']} | Tên POI = {row['rec_poi_name']} | (Total Weight: {row['total_weight']}, Số reviews toàn cục: {row['occurrences']})")


    # Khởi tạo lại chỉ mục cho DataFrame
    recommended_interactions.reset_index(drop=True, inplace=True)
    # Sắp xếp lại các cột
    recommended_interactions = recommended_interactions[['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name']]
    # Loại bỏ trùng lặp
    recommended_interactions = recommended_interactions.drop_duplicates()

    # Hiển thị DataFrame đã gộp
    return recommended_interactions.head(k)

In [37]:
# ID của người dùng mục tiêu
user_id = 371
# ID của POI mục tiêu
poi_id = 7171779

df_recommend = heuristic_recommendation(user_id, poi_id)

df_recommend

Tìm thấy 299 records POI có CÙNG KHU VỰC.
Tìm thấy 28 records POI có CÙNG DANH MỤC.
Tìm thấy 615 records POI LÂN CẬN (< 1.5km).
[Khu vực] Rút gọn còn 299 records sau khi gom nhóm.
 [Danh mục] Rút gọn còn 28 records (tính trùng lặp thể loại).
Gộp 3 danh sách (Outer Join): Tổng cộng có 720 records.
Số record VỪA cùng [khu vực] VỪA cùng [danh mục]: 720
*** Top 10 địa điểm được khuyến nghị:
      - Hạng 419: POI ID = 15528737 | Tên POI = HANA TOURIST VIP | (Total Weight: 3.0, Số reviews toàn cục: 109.0)
      - Hạng 50: POI ID = 2273094 | Tên POI = Công Ty Tnhh Du Lịch Việt Vui | (Total Weight: 3.0, Số reviews toàn cục: 94.0)
      - Hạng 170: POI ID = 7194403 | Tên POI = Ginkgo Voyage Private Day Tours | (Total Weight: 3.0, Số reviews toàn cục: 49.0)
      - Hạng 491: POI ID = 19163793 | Tên POI = 102 Saigonese | (Total Weight: 3.0, Số reviews toàn cục: 5.0)
      - Hạng 435: POI ID = 16666055 | Tên POI = Trang Thanh Travel | (Total Weight: 3.0, Số reviews toàn cục: 3.0)
      - Hạng 38: 

,user_id,poi_id,rec_poi_id,rec_poi_name
0,371.0,7171779.0,15528737,HANA TOURIST VIP
1,371.0,7171779.0,2273094,Công Ty Tnhh Du Lịch Việt Vui
2,371.0,7171779.0,7194403,Ginkgo Voyage Private Day Tours
3,371.0,7171779.0,19163793,102 Saigonese
4,371.0,7171779.0,16666055,Trang Thanh Travel
5,371.0,7171779.0,2004786,Vespa Adventures
6,371.0,7171779.0,10005057,Ben Nghe Street Food
7,371.0,7171779.0,24162332,Monkee Shisha Lounge Saigon
8,371.0,7171779.0,311087,Chợ Bến Thành
9,371.0,7171779.0,5505885,Đường Bùi Viện


# Đánh giá

In [38]:
# DataFrame các POI
pois = run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)
    RETURN poi.id
    """),
    params = {}
)

df_pois = pd.DataFrame([r.data() for r in pois])
df_pois

,poi.id
0,311103
1,2005826
2,311087
3,4542125
4,311089
...,...
3321,34509929
3322,34508995
3323,34515006
3324,34435391


In [39]:
# DataFrame các đánh giá

reviews = run(driver, textwrap.dedent("""\
    MATCH (user:User)-[review:REVIEWED]->(poi:Poi)
    RETURN user.id AS user_id, poi.id AS poi_id
    """),
    params = {}
)

df_reviews = pd.DataFrame([r.data() for r in reviews])
df_reviews

,user_id,poi_id
0,1,311103
1,2,311103
2,3,311103
3,4,311103
4,5,311103
...,...,...
23943,21311,15129760
23944,21312,15129760
23945,21313,15129760
23946,21314,15129760


In [40]:
# Nhóm theo 'user_id' và đếm số lần xuất hiện
user_counts = df_reviews.groupby('user_id').size()

# Lọc ra các người dùng có ít hơn 5 lần xuất hiện
valid_users = user_counts[user_counts >= 5].index

# Lọc DataFrame gốc dựa trên danh sách người dùng hợp lệ
filtered_df_reviews = df_reviews[df_reviews['user_id'].isin(valid_users)].copy()
filtered_df_reviews

,user_id,poi_id
50,51,311103
51,52,311103
103,104,311103
116,117,311103
118,119,311103
...,...,...
23570,481,12829864
23571,1303,12829864
23573,2886,12829864
23613,1246,9974597


In [41]:
# Chia tập dữ liệu thành 90% tập huấn luyện (training) và 10% tập kiểm tra (test)
df_train, df_test = train_test_split(filtered_df_reviews, test_size=0.1, random_state=100)

df_train

,user_id,poi_id
23463,181,9806390
2694,473,311094
3279,1177,1910195
2331,1334,10836601
8110,51,16657089
...,...,...
4403,284,311092
555,493,4542125
2215,1459,4552853
745,52,2037764


In [42]:
df_test

,user_id,poi_id
3284,1253,1910195
5049,683,552637
14873,606,10044301
1854,472,317896
2336,1415,10836601
...,...,...
1378,1006,10005057
23298,181,10388517
23338,474,1809055
545,484,311087


In [43]:
# Lấy gợi ý cho từng dòng trong tập test
df_all_retrieved = pd.DataFrame()
for index, row in df_test.iterrows():
    recommended_interactions = heuristic_recommendation(row['user_id'], row['poi_id'])
    # Nối các tương tác gợi ý với test_recommendations
    df_all_retrieved = pd.concat([df_all_retrieved, recommended_interactions], ignore_index=True)

# Loại bỏ các cột trùng lặp
df_all_retrieved = df_all_retrieved.drop_duplicates()

df_all_retrieved

Tìm thấy 823 records POI có CÙNG KHU VỰC.
Tìm thấy 1 records POI có CÙNG DANH MỤC.
Tìm thấy 586 records POI LÂN CẬN (< 1.5km).
[Khu vực] Rút gọn còn 823 records sau khi gom nhóm.
 [Danh mục] Rút gọn còn 1 records (tính trùng lặp thể loại).
Gộp 3 danh sách (Outer Join): Tổng cộng có 1008 records.
Số record VỪA cùng [khu vực] VỪA cùng [danh mục]: 1008
*** Top 10 địa điểm được khuyến nghị:
      - Hạng 35: POI ID = 1557134 | Tên POI = TNK Travel | (Total Weight: 2.0, Số reviews toàn cục: 294.0)
      - Hạng 155: POI ID = 6351606 | Tên POI = Asiana Link Travel | (Total Weight: 2.0, Số reviews toàn cục: 294.0)
      - Hạng 56: POI ID = 2037764 | Tên POI = Tháp Tài Chính Bitexco | (Total Weight: 2.0, Số reviews toàn cục: 287.0)
      - Hạng 149: POI ID = 5970625 | Tên POI = Temple Leaf Spa | (Total Weight: 2.0, Số reviews toàn cục: 270.0)
      - Hạng 547: POI ID = 15129760 | Tên POI = Phương Trang - FUTA Bus Lines | (Total Weight: 2.0, Số reviews toàn cục: 233.0)
      - Hạng 5: POI ID = 31

C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:120: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['user_id'] = recommended_interactions['user_id_region'].fillna(recommended_interactions['user_id_category']).fillna(recommended_interactions['user_id_nearby']).fillna(user_id)
C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['poi_id'] = recommended_interactions['poi_id_region'].fillna(recommended_interactions['poi_id_category']

Số record VỪA cùng [khu vực] VỪA cùng [danh mục]: 1001
*** Top 10 địa điểm được khuyến nghị:
      - Hạng 157: POI ID = 6351606 | Tên POI = Asiana Link Travel | (Total Weight: 2.0, Số reviews toàn cục: 294.0)
      - Hạng 57: POI ID = 2037764 | Tên POI = Tháp Tài Chính Bitexco | (Total Weight: 2.0, Số reviews toàn cục: 287.0)
      - Hạng 151: POI ID = 5970625 | Tên POI = Temple Leaf Spa | (Total Weight: 2.0, Số reviews toàn cục: 270.0)
      - Hạng 541: POI ID = 15129760 | Tên POI = Phương Trang - FUTA Bus Lines | (Total Weight: 2.0, Số reviews toàn cục: 233.0)
      - Hạng 6: POI ID = 311103 | Tên POI = War Remnants Museum | (Total Weight: 2.0, Số reviews toàn cục: 210.0)
      - Hạng 175: POI ID = 6721127 | Tên POI = Kim Travel | (Total Weight: 2.0, Số reviews toàn cục: 205.0)
      - Hạng 480: POI ID = 13391755 | Tên POI = Private Daily Tours | (Total Weight: 2.0, Số reviews toàn cục: 175.0)
      - Hạng 41: POI ID = 1746272 | Tên POI = Les Rives Authentic River Experience | (Total

C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:120: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['user_id'] = recommended_interactions['user_id_region'].fillna(recommended_interactions['user_id_category']).fillna(recommended_interactions['user_id_nearby']).fillna(user_id)
C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['poi_id'] = recommended_interactions['poi_id_region'].fillna(recommended_interactions['poi_id_category']

*** Top 10 địa điểm được khuyến nghị:
      - Hạng 80: POI ID = 5921457 | Tên POI = Beautiful Saigon Spa | (Total Weight: 3.0, Số reviews toàn cục: 101.0)
      - Hạng 240: POI ID = 11899009 | Tên POI = Coco Care Spa | (Total Weight: 3.0, Số reviews toàn cục: 47.0)
      - Hạng 33: POI ID = 2334935 | Tên POI = Foot Massage Salon Quỳnh Như 137 | (Total Weight: 3.0, Số reviews toàn cục: 45.0)
      - Hạng 192: POI ID = 10767864 | Tên POI = Rocky Spa | (Total Weight: 3.0, Số reviews toàn cục: 43.0)
      - Hạng 324: POI ID = 14116019 | Tên POI = Zen Spa - Foot & Body Massage | (Total Weight: 3.0, Số reviews toàn cục: 41.0)
      - Hạng 608: POI ID = 28029827 | Tên POI = Bông Spa (Branch Nguyen Canh Chan) | (Total Weight: 3.0, Số reviews toàn cục: 27.0)
      - Hạng 733: POI ID = 34495164 | Tên POI = NAP Healing & Wellness | (Total Weight: 3.0, Số reviews toàn cục: 25.0)
      - Hạng 53: POI ID = 2726268 | Tên POI = Cat Moc Spa | (Total Weight: 3.0, Số reviews toàn cục: 22.0)
      - Hạng 

...

C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:120: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['user_id'] = recommended_interactions['user_id_region'].fillna(recommended_interactions['user_id_category']).fillna(recommended_interactions['user_id_nearby']).fillna(user_id)
C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['poi_id'] = recommended_interactions['poi_id_region'].fillna(recommended_interactions['poi_id_category']

Tìm thấy 386 records POI có CÙNG DANH MỤC.
Tìm thấy 545 records POI LÂN CẬN (< 1.5km).
[Khu vực] Rút gọn còn 823 records sau khi gom nhóm.
 [Danh mục] Rút gọn còn 386 records (tính trùng lặp thể loại).
Gộp 3 danh sách (Outer Join): Tổng cộng có 1124 records.
Số record VỪA cùng [khu vực] VỪA cùng [danh mục]: 1124
*** Top 10 địa điểm được khuyến nghị:
      - Hạng 157: POI ID = 5970625 | Tên POI = Temple Leaf Spa | (Total Weight: 3.0, Số reviews toàn cục: 270.0)
      - Hạng 113: POI ID = 3749935 | Tên POI = Saigon Heritage Spa | (Total Weight: 3.0, Số reviews toàn cục: 128.0)
      - Hạng 482: POI ID = 12788024 | Tên POI = Ori Nail & Spa | (Total Weight: 3.0, Số reviews toàn cục: 86.0)
      - Hạng 742: POI ID = 19408793 | Tên POI = Mido  Spa | (Total Weight: 3.0, Số reviews toàn cục: 50.0)
      - Hạng 736: POI ID = 19259802 | Tên POI = Fame Nails | (Total Weight: 3.0, Số reviews toàn cục: 43.0)
      - Hạng 649: POI ID = 16835208 | Tên POI = The Blush Spa | (Total Weight: 3.0, Số revi

C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:120: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['user_id'] = recommended_interactions['user_id_region'].fillna(recommended_interactions['user_id_category']).fillna(recommended_interactions['user_id_nearby']).fillna(user_id)
C:\Users\admin\AppData\Local\Temp\ipykernel_20984\1452251210.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  recommended_interactions['poi_id'] = recommended_interactions['poi_id_region'].fillna(recommended_interactions['poi_id_category']

Số record VỪA cùng [khu vực] VỪA cùng [danh mục]: 977
*** Top 10 địa điểm được khuyến nghị:
      - Hạng 154: POI ID = 6351606 | Tên POI = Asiana Link Travel | (Total Weight: 2.0, Số reviews toàn cục: 294.0)
      - Hạng 56: POI ID = 2037764 | Tên POI = Tháp Tài Chính Bitexco | (Total Weight: 2.0, Số reviews toàn cục: 287.0)
      - Hạng 148: POI ID = 5970625 | Tên POI = Temple Leaf Spa | (Total Weight: 2.0, Số reviews toàn cục: 270.0)
      - Hạng 531: POI ID = 15129760 | Tên POI = Phương Trang - FUTA Bus Lines | (Total Weight: 2.0, Số reviews toàn cục: 233.0)
      - Hạng 5: POI ID = 311103 | Tên POI = War Remnants Museum | (Total Weight: 2.0, Số reviews toàn cục: 210.0)
      - Hạng 172: POI ID = 6721127 | Tên POI = Kim Travel | (Total Weight: 2.0, Số reviews toàn cục: 205.0)
      - Hạng 474: POI ID = 13391755 | Tên POI = Private Daily Tours | (Total Weight: 2.0, Số reviews toàn cục: 175.0)
      - Hạng 40: POI ID = 1746272 | Tên POI = Les Rives Authentic River Experience | (Total 

,user_id,poi_id,rec_poi_id,rec_poi_name
0,1253.0,1910195.0,1557134,TNK Travel
1,1253.0,1910195.0,6351606,Asiana Link Travel
2,1253.0,1910195.0,2037764,Tháp Tài Chính Bitexco
3,1253.0,1910195.0,5970625,Temple Leaf Spa
4,1253.0,1910195.0,15129760,Phương Trang - FUTA Bus Lines
...,...,...,...,...
1035,313.0,8290115.0,2037764,Tháp Tài Chính Bitexco
1036,313.0,8290115.0,5970625,Temple Leaf Spa
1037,313.0,8290115.0,15129760,Phương Trang - FUTA Bus Lines
1038,313.0,8290115.0,311103,War Remnants Museum


In [44]:
# Trích xuất các tương tác thực tế (ground truth) từ tất cả đánh giá
df_true_interactions = df_reviews[['user_id', 'poi_id']]
df_true_interactions

,user_id,poi_id
0,1,311103
1,2,311103
2,3,311103
3,4,311103
4,5,311103
...,...,...
23943,21311,15129760
23944,21312,15129760
23945,21313,15129760
23946,21314,15129760


In [45]:
# Lấy tất cả các thực thể liên quan bằng cách gộp tương tác thực tế và thực thể kiểm tra theo user id
df_all_relevant = pd.merge(df_test, df_true_interactions, on=['user_id'], how='inner')
# Đổi tên cột poi_id_x thành poi_id và poi_id_y thành rec_poi_id
df_all_relevant = df_all_relevant.rename(columns={'poi_id_x': 'poi_id', 'poi_id_y': 'rec_poi_id'})
# Loại bỏ các dòng có poi_id trùng với rec_poi_id
df_all_relevant = df_all_relevant[df_all_relevant['poi_id'] != df_all_relevant['rec_poi_id']]
df_all_relevant = df_all_relevant.drop_duplicates()

df_all_relevant

,user_id,poi_id,rec_poi_id
0,1253,1910195,10005057
1,1253,1910195,317893
2,1253,1910195,10836601
3,1253,1910195,1830324
5,1253,1910195,550709
...,...,...,...
1547,313,8290115,317896
1548,313,8290115,454974
1549,313,8290115,317893
1550,313,8290115,2414430


In [46]:
# Lấy tất cả các thực thể gợi ý liên quan bằng cách gộp tương tác thực tế và tương tác gợi ý
df_retrived_relevant = pd.merge(df_all_relevant, df_all_retrieved, on=['user_id', 'poi_id', 'rec_poi_id'], how='inner')
df_retrived_relevant

,user_id,poi_id,rec_poi_id,rec_poi_name
0,472,317896,2037764,Tháp Tài Chính Bitexco
1,1020,317898,8290115,Phố đi bộ Nguyễn Huệ
2,1229,10085046,317898,Notre Dame Square
3,484,5505885,311087,Chợ Bến Thành
4,484,5505885,8290115,Phố đi bộ Nguyễn Huệ
...,...,...,...,...
84,181,10388517,10044301,Công ty cổ phần xe khách Phương Trang
85,181,10388517,9806390,Bến xe Chợ Lớn
86,474,1809055,1809062,Diamond Superbowl
87,484,311087,5505885,Đường Bùi Viện


In [47]:
# Tính chỉ số độ chính xác (precision score)
relevant_retrieved = df_retrived_relevant.shape[0]
all_retrived = df_all_retrieved.shape[0]

precision = relevant_retrieved / all_retrived

print(f'Precision Score: {precision}')

Precision Score: 0.08557692307692308


In [48]:
# Tính chỉ số độ gọi lại (recall score)
relevant_retrieved = df_retrived_relevant.shape[0]
all_relevant = df_all_relevant.shape[0]

recall = relevant_retrieved / all_relevant
print(f'Recall Score: {recall}')

Recall Score: 0.06146408839779006


In [49]:
# Tính chỉ số độ phủ (coverage score)
num_recommended_pois = df_all_retrieved['rec_poi_id'].nunique()
num_all_pois = df_pois.shape[0]

coverage = num_recommended_pois / num_all_pois
print(f'Coverage Score: {coverage}')

Coverage Score: 0.059230306674684305


In [50]:
# Tính điểm F1 (F1 score)
f1 = (2 * precision * recall) / (precision + recall)
print(f'F1 Score: {f1}')

F1 Score: 0.07154340836012862


# Đóng kết nối driver

In [30]:
driver.close()